# QTTG Test Notebook

Notebook nay dung de test thu cong logic cua Bronze, Silver, Gold va Validate.
Notebook khong ghi parquet mac dinh de tranh loi `HADOOP_HOME` tren Windows.
Neu muon ghi file, hay chay bang `spark-submit` hoac moi truong da cau hinh Hadoop.

In [19]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    avg,
    coalesce,
    col,
    countDistinct,
    date_format,
    explode,
    expr,
    lit,
    max as spark_max,
    min as spark_min,
    regexp_replace,
    row_number,
    sequence,
    sum as spark_sum,
    to_date,
    trim,
    when,
)
from pyspark.sql.window import Window

from schemas import MASTER_SCHEMA, DETAIL_SCHEMA

## Config

Sua 2 duong dan nay neu may ban khac.

In [20]:
RAW_DIR = r"C:/Users/thinkbook123/Datahub/output/raw_qttg_1m"
LAKE_DIR = r"C:/Users/thinkbook123/Datahub/output/lake"

MASTER_CSV = f"{RAW_DIR}/RAW_QTTG_BHXH.csv"
DETAIL_CSV = f"{RAW_DIR}/RAW_QTTG_BHXH_DETAIL.csv"

print("MASTER_CSV =", MASTER_CSV)
print("DETAIL_CSV =", DETAIL_CSV)
print("master exists:", os.path.exists(MASTER_CSV))
print("detail exists:", os.path.exists(DETAIL_CSV))

if not os.path.exists(MASTER_CSV) or not os.path.exists(DETAIL_CSV):
    raise FileNotFoundError("Sua lai RAW_DIR truoc khi chay tiep.")

MASTER_CSV = C:/Users/thinkbook123/Datahub/output/raw_qttg_1m/RAW_QTTG_BHXH.csv
DETAIL_CSV = C:/Users/thinkbook123/Datahub/output/raw_qttg_1m/RAW_QTTG_BHXH_DETAIL.csv
master exists: True
detail exists: True


In [21]:
spark = (
    SparkSession.builder.appName("QTTG_Test_Notebook")
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
    .getOrCreate()
)

spark

## 1. Bronze

In [22]:
df_master_bronze = (
    spark.read.option("header", "true")
    .schema(MASTER_SCHEMA)
    .csv(MASTER_CSV)
)

df_detail_bronze = (
    spark.read.option("header", "true")
    .schema(DETAIL_SCHEMA)
    .csv(DETAIL_CSV)
)

master_count = df_master_bronze.count()
detail_count = df_detail_bronze.count()

print("master rows:", master_count)
print("detail rows:", detail_count)
df_master_bronze.printSchema()

master rows: 142857
detail rows: 1000000
root
 |-- ID: long (nullable = true)
 |-- NLD_ID: long (nullable = true)
 |-- SO_SO_BHXH: string (nullable = true)
 |-- THANG_BD: string (nullable = true)
 |-- THANG_KT: string (nullable = true)
 |-- TT_TG_BHXH: string (nullable = true)
 |-- DT_TG_BHXH: string (nullable = true)
 |-- NAM_TG_BHXH: integer (nullable = true)
 |-- THANG_TG_BHXH: integer (nullable = true)
 |-- NAM_TG_BHXH_BB: integer (nullable = true)
 |-- THANG_TG_BHXH_BB: integer (nullable = true)
 |-- TT_TG_BHTN: string (nullable = true)
 |-- DT_TG_BHTN: string (nullable = true)
 |-- NAM_TG_BHTN: integer (nullable = true)
 |-- THANG_TG_BHTN: integer (nullable = true)
 |-- TT_TG_BHYT: string (nullable = true)
 |-- DT_TG_BHYT: string (nullable = true)
 |-- NAM_TG_BHYT: integer (nullable = true)
 |-- THANG_TG_BHYT: integer (nullable = true)
 |-- NAM_NO_BHXH: integer (nullable = true)
 |-- THANG_NO_BHXH: integer (nullable = true)
 |-- NAM_NO_BHTN: integer (nullable = true)
 |-- THANG_N

In [23]:
EXPECTED_MASTER_ROWS = 142857
EXPECTED_DETAIL_ROWS = 1000000

assert master_count == EXPECTED_MASTER_ROWS, f"master rows sai: {master_count}"
assert detail_count == EXPECTED_DETAIL_ROWS, f"detail rows sai: {detail_count}"
print("Bronze check: PASS")

Bronze check: PASS


## 2. Silver

In [24]:
df_cleaned_master = (
    df_master_bronze.withColumn("SO_SO_BHXH", trim(col("SO_SO_BHXH")))
    .withColumn("THANG_BD", regexp_replace(col("THANG_BD"), r"[^0-9]", ""))
    .withColumn("THANG_KT", regexp_replace(col("THANG_KT"), r"[^0-9]", ""))
)

window_spec = Window.partitionBy("SO_SO_BHXH").orderBy(
    col("CREATED_AT").desc(), col("ID").desc()
)

df_silver_master = (
    df_cleaned_master.withColumn("rn", row_number().over(window_spec))
    .withColumn("IS_DELETED", when(col("rn") == 1, 0).otherwise(1))
    .drop("rn")
)

df_cleaned_detail = (
    df_detail_bronze.withColumn("MA_DON_VI", trim(col("MA_DON_VI")))
    .withColumn("TU_THANG", regexp_replace(col("TU_THANG"), r"[^0-9]", ""))
    .withColumn("DEN_THANG", regexp_replace(col("DEN_THANG"), r"[^0-9]", ""))
    .withColumn("MUC_LUONG", coalesce(col("MUC_LUONG"), lit(0)))
)

df_active_master = df_silver_master.filter(col("IS_DELETED") == 0).select(
    col("ID").alias("M_ID"), col("NLD_ID").alias("M_NLD_ID")
)

df_silver_detail = df_cleaned_detail.join(
    df_active_master,
    (df_cleaned_detail["MASTER_ID"] == df_active_master["M_ID"])
    & (df_cleaned_detail["NLD_ID"] == df_active_master["M_NLD_ID"]),
    "inner",
).drop("M_ID", "M_NLD_ID")

print("silver active master:", df_silver_master.filter(col("IS_DELETED") == 0).count())
print("silver detail:", df_silver_detail.count())
df_silver_master.select("SO_SO_BHXH", "CREATED_AT", "IS_DELETED").show(10, False)

silver active master: 21838
silver detail: 169658
+----------+--------------------------+----------+
|SO_SO_BHXH|CREATED_AT                |IS_DELETED|
+----------+--------------------------+----------+
|7000000001|2026-01-01 08:00:01.000000|0         |
|7000000002|2026-01-01 08:00:02.000000|0         |
|7000000003|2026-01-01 08:00:03.000000|0         |
|7000000004|2026-01-01 08:00:04.000000|0         |
|7000000005|2026-01-01 08:00:05.000000|0         |
|7000000006|2026-01-01 08:00:06.000000|0         |
|7000000007|2026-01-01 08:00:07.000000|0         |
|7000000008|2026-01-01 08:00:08.000000|0         |
|7000000009|2026-01-01 08:00:09.000000|0         |
|7000000010|2026-01-01 08:00:10.000000|0         |
+----------+--------------------------+----------+
only showing top 10 rows


In [25]:
duplicate_persons = (
    df_silver_master.filter(col("IS_DELETED") == 0)
    .groupBy("SO_SO_BHXH")
    .count()
    .filter(col("count") > 1)
    .count()
)

invalid_month_range = df_silver_detail.filter(col("TU_THANG") > col("DEN_THANG")).count()

print("duplicate_persons =", duplicate_persons)
print("invalid_month_range =", invalid_month_range)
assert duplicate_persons == 0, "Silver bi trung nguoi active"
assert invalid_month_range == 0, "Silver co TU_THANG > DEN_THANG"
print("Silver check: PASS")

duplicate_persons = 0
invalid_month_range = 0
Silver check: PASS


## 3. Gold

In [26]:
df_dim_thang = (
    df_silver_detail
    .agg(
        spark_min(to_date(col("TU_THANG"), "yyyyMM")).alias("min_date"),
        spark_max(to_date(col("DEN_THANG"), "yyyyMM")).alias("max_date"),
    )
    .select(explode(sequence(col("min_date"), col("max_date"), expr("interval 1 month"))).alias("month_date"))
    .select(date_format(col("month_date"), "yyyyMM").alias("THANG_ID"))
)

df_gold = (
    df_dim_thang
    .join(
        df_silver_detail,
        (col("THANG_ID") >= df_silver_detail["TU_THANG"])
        & (col("THANG_ID") <= df_silver_detail["DEN_THANG"]),
        "inner",
    )
    .join(
        df_silver_master.filter(col("IS_DELETED") == 0).select(
            col("ID").alias("M_ID"), col("SO_SO_BHXH")
        ),
        col("MASTER_ID") == col("M_ID"),
        "inner",
    )
    .groupBy("THANG_ID")
    .agg(
        countDistinct("SO_SO_BHXH").alias("SO_NGUOI_THAM_GIA"),
        countDistinct("MA_DON_VI").alias("SO_DON_VI"),
        spark_sum("MUC_LUONG").alias("TONG_QUY_LUONG"),
        avg(when(col("MUC_LUONG") > 0, col("MUC_LUONG"))).alias("LUONG_BINH_QUAN"),
        countDistinct(when(col("MUC_LUONG") == 0, col("SO_SO_BHXH"))).alias("SO_NGUOI_LUONG_0"),
    )
)

print("gold rows:", df_gold.count())
df_gold.orderBy("THANG_ID").show(20, False)

gold rows: 629
+--------+-----------------+---------+---------------+----------------+----------------+
|THANG_ID|SO_NGUOI_THAM_GIA|SO_DON_VI|TONG_QUY_LUONG |LUONG_BINH_QUAN |SO_NGUOI_LUONG_0|
+--------+-----------------+---------+---------------+----------------+----------------+
|197402  |40               |40       |165500000.0000 |5338709.67741935|9               |
|197403  |84               |84       |318650000.0000 |5223770.49180328|23              |
|197404  |123              |121      |503750000.0000 |5857558.13953488|37              |
|197405  |165              |163      |691600000.0000 |6013913.04347826|50              |
|197406  |190              |188      |774400000.0000 |5736296.29629630|55              |
|197407  |225              |222      |943500000.0000 |5788343.55828221|62              |
|197408  |260              |255      |1122400000.0000|5815544.04145078|67              |
|197409  |285              |281      |1303800000.0000|6036111.11111111|69              |
|19741

In [27]:
duplicate_gold_months = (
    df_gold.groupBy("THANG_ID")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("duplicate_gold_months =", duplicate_gold_months)
assert duplicate_gold_months == 0, "Gold bi trung THANG_ID"
print("Gold check: PASS")

duplicate_gold_months = 0
Gold check: PASS


## 4. Validate

In [28]:
orphan_details = df_silver_detail.join(
    df_silver_master,
    df_silver_detail["MASTER_ID"] == df_silver_master["ID"],
    "left_anti",
).count()

duplicate_persons = (
    df_silver_master.filter(col("IS_DELETED") == 0)
    .groupBy("SO_SO_BHXH")
    .count()
    .filter(col("count") > 1)
    .count()
)

invalid_month_range = df_silver_detail.filter(col("TU_THANG") > col("DEN_THANG")).count()

duplicate_gold_months = (
    df_gold.groupBy("THANG_ID")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("orphan_details =", orphan_details)
print("duplicate_persons =", duplicate_persons)
print("invalid_month_range =", invalid_month_range)
print("duplicate_gold_months =", duplicate_gold_months)

assert orphan_details == 0, "Silver detail co dong mo coi"
assert duplicate_persons == 0, "Silver co trung nguoi active"
assert invalid_month_range == 0, "Silver co TU_THANG > DEN_THANG"
assert duplicate_gold_months == 0, "Gold bi trung THANG_ID"
print("Validate check: PASS")

orphan_details = 0
duplicate_persons = 0
invalid_month_range = 0
duplicate_gold_months = 0
Validate check: PASS


## Optional Write

Chi dung cell nay neu moi truong Spark cua ban ghi parquet duoc.

In [29]:
# BO comment neu ban muon ghi parquet.
# df_master_bronze.write.mode("overwrite").parquet(f"{LAKE_DIR}/bronze/master")
# df_detail_bronze.write.mode("overwrite").parquet(f"{LAKE_DIR}/bronze/detail")
# df_silver_master.write.mode("overwrite").parquet(f"{LAKE_DIR}/silver/master")
# df_silver_detail.write.mode("overwrite").parquet(f"{LAKE_DIR}/silver/detail")
# df_gold.write.mode("overwrite").parquet(f"{LAKE_DIR}/gold")

In [30]:
spark.stop()